In [ ]:
import networkx as nx  # networkx used for making graph
import matplotlib.pyplot as plt  # matplotlib used for plotting data in visual graphic
import matplotlib.image as mpimg  # used to show background image
import time  # for performance benchmarking


def import_data() -> tuple[dict, list]:
    """imports two files, returns a tuple: dict of locations, list of adjacencies"""

    # in order to filter counties to the continental US, the following min / max values are used
    # original data file includes AK, HI, PR, Guam, American Samoa, etc that should be ignored
    # This essentially creates a box around the continental US
    MIN_LONG = -125  # western edge
    MAX_LONG = -65  # eastern edge
    MIN_LAT = 24  # southern edge
    MAX_LAT = 50  # northern edge

    # Import county names and location data (lat and long) from file ------------------------------
    # This represents Nodes on a graph
    county_locations = {}
    county_loc_file = open("../data/county_data.csv", "r")
    co_loc_header = next(county_loc_file)  # skips first line

    line_number = 0
    for line in county_loc_file:
        fips_code, name, longitude, latitude = line.strip().split(",")

        # filters out counties outside of continental US
        if (
            MIN_LONG < float(longitude) < MAX_LONG
            and MIN_LAT < float(latitude) < MAX_LAT
        ):
            county_locations[fips_code] = {
                "fips_code": fips_code,
                "county_name": name,
                "longitude": longitude,
                "latitude": latitude,
            }
        line_number += 1

    county_loc_file.close()
    # print("location file counties imported: ", line_number)

    # Import county adjacency (neighbor borders) data from file, add lat and long info ------------
    # This represents Edges on a graph
    county_adj_data = []
    county_adj_file = open("../data/county_data.txt", "r")
    co_adj_header = next(county_adj_file)  # skips first line

    line_number = 0
    key_errors = 0
    for line in county_adj_file:
        county_name, county_geoid, neighbor_name, neighbor_geoid, border_length = (
            line.strip().split("|")
        )

        county_lat = (
            "25"  # these are sentinel values to show when the data is not present
        )
        county_long = "-66"  # sentinel value
        # There could be GEOIDs missing from the location data file
        # If not there, this avoids error
        try:
            county_lat = county_locations[county_geoid]["latitude"]
            county_long = county_locations[county_geoid]["longitude"]
        except KeyError as e:
            # KeyError occurs when a Geoid (FIPS code) is in the Adj file but not in the location file
            key_errors += 1
            print(
                f"KeyError: {e} County Geoid from Adj file not found in lat/long file"
            )
        else:
            # this runs only if the try block succeeds
            county_adj_data.append(
                {
                    "co_name": county_name,
                    "co_geoid": county_geoid,
                    "co_lat": county_lat,
                    "co_long": county_long,
                    "neighbor_name": neighbor_name,
                    "neighbor_geoid": neighbor_geoid,
                    "border_length": border_length,
                }
            )
            line_number += 1

    county_adj_file.close()
    # print("Number of KeyErrors: ", key_errors)
    # print("Adjacency file counties imported: ", line_number)

    # prints five example lines for each data set
    # print("Example of County Locations: ")
    # for key, value in list(county_locations.items())[:5]:
    #     print(key, value)
    # Example data:
    # 01059 {'fips_code': '01059', 'county_name': 'Franklin', 'longitude': '-87.843283', 'latitude': '34.44238135'}

    # print("Example County Adjacency Data: ")
    # for i in range(5):
    #     print(county_adj_data[i])
    # Example data:
    # {'co_name': 'Autauga County, AL', 'co_geoid': '01001', 'co_lat': '32.53514228', 'co_long': '-86.64289976',
    #           'neighbor_name': 'Chilton County, AL', 'neighbor_geoid': '01021', 'border_length': '52284'}

    return county_locations, county_adj_data

In [ ]:
import networkx as nx  # networkx used for making graph
import matplotlib.pyplot as plt  # matplotlib used for plotting data in visual graphic
import matplotlib.image as mpimg  # used to show background image
import time  # for performance benchmarking


def import_state_data(state_code) -> tuple[dict, list]:
    """
    imports two files, returns a tuple: dict of locations, list of adjacencies
    restrict which state you are creating a graph for by pass in STATE

    Only counties in `state_code` are returned, so the graph built from them is the state's
    county graph and nothing else. Adjacency comes from the official Census County Adjacency
    File (2024), which unlike ../data/county_data.txt also covers AK and HI.
    """

    # Import county adjacency (neighbor borders) data from file -----------------------------------
    # This represents Edges on a graph. Each county also has a row pairing it with itself;
    # those rows give us every county in the state (including ones with no in-state neighbor)
    # and are not edges.
    state_counties = {}
    county_adj_data = []
    county_adj_file = open("../data/county_adjacency2024.txt", "r")
    co_adj_header = next(county_adj_file)  # skips first line

    for line in county_adj_file:
        county_name, county_geoid, neighbor_name, neighbor_geoid = line.rstrip("\n").split("|")[:4]
        source = county_name.strip().split(",")
        destination = neighbor_name.strip().split(",")
        if source[-1].strip() != state_code:
            continue
        state_counties[county_geoid] = county_name
        if county_geoid == neighbor_geoid or destination[-1].strip() != state_code:
            continue  # self row, or a neighbor across the state line
        county_adj_data.append(
            {
                "co_name": county_name,
                "co_geoid": county_geoid,
                "neighbor_name": neighbor_name,
                "neighbor_geoid": neighbor_geoid,
            }
        )
    county_adj_file.close()

    # Import county lat/long, for drawing only ----------------------------------------------------
    # county_data.csv has no AK or HI rows, so a county may have no position (None).
    known = {}
    county_loc_file = open("../data/county_data.csv", "r")
    co_loc_header = next(county_loc_file)  # skips first line
    for line in county_loc_file:
        fips_code, name, longitude, latitude = line.strip().split(",")
        known[fips_code] = (longitude, latitude)
    county_loc_file.close()

    county_locations = {}
    for fips_code, name in state_counties.items():
        longitude, latitude = known.get(fips_code, (None, None))
        county_locations[fips_code] = {
            "fips_code": fips_code,
            "county_name": name,
            "longitude": longitude,
            "latitude": latitude,
        }

    # Example data:
    # county_locations['01059'] = {'fips_code': '01059', 'county_name': 'Franklin County, AL',
    #                              'longitude': '-87.843283', 'latitude': '34.44238135'}
    # county_adj_data[0] = {'co_name': 'Autauga County, AL', 'co_geoid': '01001',
    #                       'neighbor_name': 'Chilton County, AL', 'neighbor_geoid': '01021'}

    return county_locations, county_adj_data


In [ ]:
import pandas as pd

STATES = [
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA", 
    "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD", 
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ", 
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC", 
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY"
]

data = {"State" : [], "Algorithm": [], "Colors Used" : [], "Runtime" : []}
df = pd.DataFrame(data)
counter = 0

for state in STATES:

    G = nx.Graph()

    # import data from files
    county_locations, county_adj_data = import_state_data(state)

    # Create graph nodes (each county) and node positions (lat & long) --------
    positions = {}
    for co_geoid, county in county_locations.items():
        G.add_node(co_geoid)
        if county["latitude"] is not None:  # AK and HI have no lat/long in county_data.csv
            positions[co_geoid] = (float(county["longitude"]), float(county["latitude"]))

    # Creating graph edges based on borders in adjacency data -----------------
    for border in county_adj_data:
        edge_start = border["co_geoid"]
        edge_end = border["neighbor_geoid"]
        G.add_edge(edge_start, edge_end)


    ALGORITHMS = [
        'largest_first',
        'random_sequential',
        'smallest_last',
        'independent_set',
        'connected_sequential_bfs',
        'connected_sequential_dfs',
        'connected_sequential', # (alias for the previous strategy)
        'saturation_largest_first',
        'DSATUR', # (alias for the previous strategy)
    ]

    for index, algo in enumerate(ALGORITHMS):
        # Benchmarking start
        start_time = time.perf_counter()

        # Graph coloring here - select algorithm here
        node_color_label_dict = nx.coloring.greedy_color(
            G, strategy=algo, interchange=False
        )

        end_time = time.perf_counter()

        number_of_colors = max(node_color_label_dict.values()) + 1
        
        df.loc[counter] = [state, algo, number_of_colors, end_time - start_time]
        counter += 1
        
    print(f"Finished running: {state} ({G.number_of_nodes()} counties, {G.number_of_edges()} edges)")
        
        
df.to_csv("../results/algorithms_by_state.csv")

In [ ]:
import pandas as pd

G = nx.Graph()

# import data from files
county_locations, county_adj_data = import_data()

# Create graph nodes (each county) and node positions (lat & long) --------
positions = {}
for co_geoid, county in county_locations.items():
    G.add_node(co_geoid)
    lat = float(county["latitude"])
    long = float(county["longitude"])
    positions[co_geoid] = (long, lat)

# Creating graph edges based on borders in adjacency data -----------------
for border in county_adj_data:
    edge_start = border["co_geoid"]
    edge_end = border["neighbor_geoid"]
    G.add_edge(edge_start, edge_end)


ALGORITHMS = [
    'largest_first',
    'random_sequential',
    'smallest_last',
    'independent_set',
    'connected_sequential_bfs',
    'connected_sequential_dfs',
    'connected_sequential', # (alias for the previous strategy)
    'saturation_largest_first',
    'DSATUR', # (alias for the previous strategy)
]

data = {"Algorithm": [], "Colors Used" : [], "Runtime" : []}
df = pd.DataFrame(data)

for index, algo in enumerate(ALGORITHMS):
    # Benchmarking start
    start_time = time.perf_counter()

    # Graph coloring here - select algorithm here
    node_color_label_dict = nx.coloring.greedy_color(
        G, strategy=algo, interchange=False
    )

    end_time = time.perf_counter()

    number_of_colors = max(node_color_label_dict.values()) + 1
    
    df.loc[index] = [algo, number_of_colors, end_time - start_time]
    
    # print("Algorith Used: ", algo)
    # print("Number of colors used for Map Coloring: ", number_of_colors)
    # print(f"Elapsed time in seconds: {(end_time - start_time):.4f}")
    
print(df)

In [ ]:
# create empty graph
G = nx.Graph()

# import data from files
county_locations, county_adj_data = import_data()

# Create graph nodes (each county) and node positions (lat & long) --------
positions = {}
for co_geoid, county in county_locations.items():
    G.add_node(co_geoid)
    lat = float(county["latitude"])
    long = float(county["longitude"])
    positions[co_geoid] = (long, lat)

# Creating graph edges based on borders in adjacency data -----------------
for border in county_adj_data:
    edge_start = border["co_geoid"]
    edge_end = border["neighbor_geoid"]
    G.add_edge(edge_start, edge_end)

# Graph Coloring in this section ------------------------------------------
# print("Now calculating graph coloring.....")

# these are the built-in greedy coloring strategies.  Will also accept a custom function
# https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.coloring.greedy_color.html
"""
'largest_first'
'random_sequential'
'smallest_last'
'independent_set'
'connected_sequential_bfs'
'connected_sequential_dfs'
'connected_sequential' # (alias for the previous strategy)
'saturation_largest_first'
'DSATUR' # (alias for the previous strategy)
"""

# Benchmarking start
start_time = time.perf_counter()

# Graph coloring here - select algorithm here
node_color_label_dict = nx.coloring.greedy_color(
    G, strategy="DSATUR", interchange=False
)

end_time = time.perf_counter()

number_of_colors = max(node_color_label_dict.values()) + 1
print("Number of colors used for Map Coloring: ", number_of_colors)
print(f"Elapsed time in seconds: {(end_time - start_time):.4f}")

# maps color number labels to actual display colors for visualization
color_map_dict = {
    0: "tab:red",
    1: "tab:orange",
    2: "tab:olive",
    3: "tab:green",
    4: "tab:blue",
    5: "tab:purple",
    6: "tab:brown",
    7: "tab:gray",
    8: "tab:black",
}

# creates a list of the actual visualization colors in the correct order
node_viz_colors = [
    color_map_dict[node_color_label_dict[node]] for node in G.nodes()
]

# Below here all about drawing & displaying graph -------------------------

# visualization graphics options
options = {
    "font_size": 6,  # font size for labels
    "node_size": 10,  # size of node
    # "node_color": "yellow",   # node color (not to be used when dynamically coloring)
    "edgecolors": "black",
    "linewidths": 0,  # node border line width with 0=no border
    "width": 1,
}

# draw the graph

# One of the two following options should be used:
# nx.draw_networkx(G, positions, **options)             # allows labeling nodes and more options.  Slower
nx.draw(
    G, positions, node_color=node_viz_colors, **options
)  # simple drawing function with no node labels.  Faster

# if we wanted to place a image in the background, this would be used -----
# load image from file for background
# image = mpimg.imread('./images/background_image.jpg')

# x_right = -73.70878
# y_top = 40.92133
# x_left = -74.301217
# y_bottom = 40.482218

# ax = plt.gca()
# ax.imshow(image, aspect='auto', extent=[x_left, x_right, y_bottom, y_top] )

plt.gca().set_aspect(1.2)  # sets output aspect ratio as 1:1
# ax.margins(0.05)    # sets a margin around the plotted area
# plt.axis(True)
plt.show()


In [ ]:
# Benchmarking start
start_time = time.perf_counter()

# Graph coloring here - select algorithm here
node_color_label_dict = nx.coloring.greedy_color(
    G, strategy="largest_first", interchange=False
)

end_time = time.perf_counter()

number_of_colors = max(node_color_label_dict.values()) + 1
print("Number of colors used for Map Coloring: ", number_of_colors)
print(f"Elapsed time in seconds: {(end_time - start_time):.4f}")

color_map_dict = {
    0: "tab:red",
    1: "tab:orange",
    2: "tab:olive",
    3: "tab:green",
    4: "tab:blue",
    5: "tab:purple",
    6: "tab:brown",
    7: "tab:gray",
    8: "tab:black",
}

node_viz_colors = [
    color_map_dict[node_color_label_dict[node]] for node in G.nodes()
]

# visualization graphics options
options = {
    "font_size": 6,  # font size for labels
    "node_size": 5,  # size of node
    # "node_color": "yellow",   # node color (not to be used when dynamically coloring)
    "edgecolors": "black",
    "linewidths": 0,  # node border line width with 0=no border
    "width": 0.25,
}

nx.draw(
    G, positions, node_color=node_viz_colors, **options
)  # simple drawing function with no node labels.  Faster

plt.gca().set_aspect(1.2)  # sets output aspect ratio as 1:1
plt.show()